# 07 — Gate AB-2: MGA pilots (aggregate + guarded) at both budget levels (R)

Kernel `R (y2y)`; live internet. The parent's estimator `mga_maxham_v1` (k = 50, g = 5%) on the
reference formulation `s0_ssp585_theta5` at **level A and level B**, each in **both semantics**:
the aggregate 5% band (the Claim-A estimand; verdict rule v2 in 08) and the per-block-floor
guarded band (capture_b ≥ 0.95·anchor_b; the applied headline). Level A also gets the E4 f(g)
grid (g = 2%, 10%). Anchors are re-solved inside the MGA machinery and checked against 06's
certified engine anchors (drift ≤ 1e-3). Outputs → `runs/ab_l/<level>/s0_ssp585_theta5/`:
`anchor.tif`, `formulation_meta.json`, `mga_g05.tif`, `mga_guard_g05.tif` (+ `mga_g02/g10` at A),
certificates per sweep. Resumable per artifact. Expect minutes per sweep at 85k PU.

In [1]:
# ---- setup ---------------------------------------------------------------------------------------------
ANALYSIS <- "ab_y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
HERE <- file.path(PROJ, "analyses", "alberta_prioritization")
mpath <- file.path(PROJ, "input_data", "aligned_stack_ab", "manifest.json")
stopifnot(file.exists(mpath))
SC  <- jsonlite::read_json(file.path(HERE, "spec", "scenarios_ab_v1.json"))
LV  <- jsonlite::read_json(file.path(HERE, "spec", "ab_budget_levels_v1.json"))$levels
BLOCKS <- lapply(SC$`_meta`$blocks, unlist)
stopifnot(setequal(names(BLOCKS), c("core_habitat", "connectivity", "carbon", "biodiversity")))
RUNS_REL <- "analyses/alberta_prioritization/runs/ab_l"
REF <- list(id = "s0_ssp585_theta5", scen = "S0_balanced")
K <- 50; G <- 0.05; FLOOR_G <- 0.05
FG_GRID_LEVEL <- "A"; FG_EXTRA <- c(0.02, 0.10)   # g=10% at A only
FG_BOTH <- c(0.02)                                   # g=2% at BOTH levels (M9: nesting needs non-empty cores)
cat("reference", REF$id, "| k", K, "| g", G, "| floors on", paste(names(BLOCKS), collapse = ", "), "\n")

ctx585 <- pr_setup(mpath, PROJ)
ctx585 <- modifyList(ctx585, pr_ingest(ctx585))
built_ctx <- function(level) {
  b <- pr_override(ctx585, budget_pct = LV[[level]]$budget_pct,
                   targets = SC[[REF$scen]]$targets, feature_weight_multipliers = SC[[REF$scen]]$weights,
                   results_dir = file.path(RUNS_REL, level, REF$id), results_subdir = "mga_build",
                   solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
  b <- modifyList(b, pr_planning_units(b))
  b <- modifyList(b, pr_weights(b)); b <- modifyList(b, pr_targets(b)); b <- modifyList(b, pr_penalty_matrices(b))
  bp <- pr_build_problem(b); b$p <- bp$p; b$solve_params <- bp$solve_params
  b
}

reference s0_ssp585_theta5 | k 50 | g 0.05 | floors on core_habitat, connectivity, carbon, biodiversity 
prioritizr 8.1.0 | terra 1.9.34 | analysis=ab_y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/ab_y2y
ingested 35 features (8 continuous + 27 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 35 features to total=100000 each (scale-invariant conditioning)


In [2]:
# ---- per level: anchor (checked vs 06), aggregate sweep, guarded sweep, f(g) grid at A -------------------
run_level <- function(level) {
  cd <- file.path(PROJ, RUNS_REL, level, REF$id)
  eng <- file.path(cd, "anchor", "run_summary.json")
  stopifnot("06's engine anchor missing for this level -- run 06 first" = file.exists(eng))
  z06 <- as.numeric(unlist(jsonlite::read_json(eng)$solver_provenance$objective))[1]
  need <- c(file.path(cd, "mga_g05.tif"), file.path(cd, "mga_guard_g05.tif"),
            file.path(cd, sprintf("mga_g%02d.tif", round(100 * FG_BOTH))),
            if (level == FG_GRID_LEVEL) file.path(cd, sprintf("mga_g%02d.tif", round(100 * FG_EXTRA))))
  if (all(file.exists(need))) { cat(sprintf("level %s: all sweeps exist -- skipped\n", level)); return(invisible(NULL)) }
  actx <- built_ctx(level)
  cm <- mga_compile(actx)
  anchor <- mga_anchor(cm, opt_gap = 1e-4)
  rel <- abs(anchor$z - z06) / abs(z06)
  stopifnot("MGA anchor drifted > 1e-3 from 06's certified engine anchor -- STOP" = rel <= 1e-3)
  cat(sprintf("level %s anchor %.6f vs engine %.6f (rel %.1e)\n", level, anchor$z, z06, rel))
  if (!file.exists(file.path(cd, "anchor.tif"))) {
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r))
    v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
    terra::writeRaster(r, file.path(cd, "anchor.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255,
                       gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  }
  jsonlite::write_json(list(formulation_id = REF$id, level = level, budget_cells = LV[[level]]$budget_cells,
                            estimator = "mga_maxham_v1", anchor_objective = anchor$z, anchor_bound = anchor$bound,
                            anchor_gap = anchor$gap, anchor_runtime_s = anchor$runtime, engine_anchor_objective = z06,
                            anchor_rel_drift = rel, k = K, g = G, floor_g = FLOOR_G, blocks = BLOCKS,
                            opt_gap = 1e-4, mip_gap_dist = 0.01, time_limit_iter = 900,
                            created_utc = format(Sys.time(), tz = "UTC")),
                       file.path(cd, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
  sweep <- function(tag, g, floors = NULL) {
    if (file.exists(file.path(cd, sprintf("mga_%s.tif", tag)))) { cat(sprintf("   %s exists -- skipped\n", tag)); return(invisible(NULL)) }
    gen <- mga_generate(cm, anchor, g = g, k = K, floors = floors)
    mga_write(gen, cm, actx$cost, cd, tag)
  }
  sweep("g05", G)
  sweep("guard_g05", G, floors = list(ctx = actx, blocks = BLOCKS, g = FLOOR_G))
  for (g in FG_BOTH) sweep(sprintf("g%02d", round(100 * g)), g)
  if (level == FG_GRID_LEVEL) for (g in setdiff(FG_EXTRA, FG_BOTH)) sweep(sprintf("g%02d", round(100 * g)), g)
  invisible(NULL)
}
t0 <- proc.time()[["elapsed"]]
for (level in c("A", "B")) {
  cat(sprintf("\n===================== %s @ level %s =====================\n", REF$id, level))
  run_level(level)
  cat(sprintf("== elapsed %.1f min\n", (proc.time()[["elapsed"]] - t0) / 60))
}
cat("\nAB-2 SWEEPS COMPLETE -- next: 08_ab2_analysis.ipynb (kernel y2y-geo)\n")


===================== s0_ssp585_theta5 @ level A =====================
level A: all sweeps exist -- skipped
== elapsed 0.0 min

===================== s0_ssp585_theta5 @ level B =====================
  override budget_pct       -> 0.3877932
  override targets          -> irrecoverable_carbon_m_soc=0.322
  override feature_weight_multipliers -> climate_type_macrorefugia=1.038, transboundary_connectivity=0.30626, climate_corridors=1.5978, irrecoverable_carbon_m_soc=0.219191, irrecoverable_carbon_biomass=0.171899, aoh_richness_birds=1.38582, aoh_richness_mammals=2.28103
  override results_dir      -> analyses/alberta_prioritization/runs/ab_l/B/s0_ssp585_theta5
  override results_subdir   -> mga_build
  override solver           -> gurobi
  override decision_type    -> binary
  override opt_gap          -> 1e-04
  override portfolio_n      -> 1
  EFFECTIVE targets: irrecoverable_carbon_m_soc=0.322 | weight multipliers: climate_type_macrorefugia=1.038, transboundary_connectivity=0.30626, cl

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (35 total)
│└•planning units:
│ ├•data:       <SpatRaster> (85133 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 33014)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.322 and 1)
││└•weights:    continuous values (between 0.03703704 and 2.281027)
│├•constraints: 
││└•1:          locked in constraints (27972 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 85168 cols (85133 pu + 35 aux) x 36 rows | 27972 locked pu | modelsense min
anchor: objective 5.022023 (bound 5.022023, gap 0.00e+00) | 33,014 selected | 1 s
level B anchor 5.022023 vs engine 5.022000 (rel 4.6e-06)
   g05 exists -- skipped
   guard_g05 exists -- skipped
band wall appended: obj0 . x <= 5.122464  (g = 0.02 on z* = 5.022023)
g=0.02 iter 01/50: band 5.122432 (+2.00% of z*) OK | ham(anchor) 9,408 | 1 s
g=0.02 iter 02/50: band 5.122421 (+2.00% of z*) OK | ham(anchor) 8,602 | 2 s
g=0.02 iter 03/50: band 5.122461 (+2.00% of z*) OK | ham(anchor) 7,960 | 2 s
g=0.02 iter 04/50: band 5.122446 (+2.00% of z*) OK | ham(anchor) 7,178 | 1 s
g=0.02 iter 05/50: band 5.122445 (+2.00% of z*) OK | ham(anchor) 6,566 | 2 s
g=0.02 iter 06/50: band 5.122457 (+2.00% of z*) OK | ham(anchor) 5,436 | 1 s
g=0.02 iter 07/50: band 5.122453 (+2.00% of z*) OK | ham(anchor) 6,764 | 1 s
g=0.02 iter 08/50: band 5.122440 (+2.00% of z*) OK | ham(anchor) 7,608 | 1 s
g=0.02 iter 09/50: band 5.122462 

In [3]:
# ---- integrity summary ------------------------------------------------------------------------------------
for (level in c("A", "B")) {
  cd <- file.path(PROJ, RUNS_REL, level, REF$id)
  for (tag in c("g05", "guard_g05", "g02", "g10")) {
    csv <- file.path(cd, sprintf("certificates_%s.csv", tag))
    if (!file.exists(csv)) next
    ce <- read.csv(csv)
    cat(sprintf("%s %-10s %2d members | band_ok %s | dup %d | time-limited %d | %5.1f min | max ham %s\n",
                level, tag, nrow(ce), if (all(ce$band_ok)) "ALL" else "VIOLATED", sum(ce$duplicate),
                sum(ce$status == "TIME_LIMIT"), sum(ce$runtime_s) / 60, format(max(ce$hamming_to_anchor), big.mark = ",")))
  }
}

A g05        50 members | band_ok ALL | dup 0 | time-limited 0 |   1.1 min | max ham 20,126
A guard_g05  50 members | band_ok ALL | dup 0 | time-limited 0 |   1.5 min | max ham 20,126
A g02        50 members | band_ok ALL | dup 0 | time-limited 0 |   1.1 min | max ham 15,950
A g10        50 members | band_ok ALL | dup 2 | time-limited 0 |   0.5 min | max ham 20,166
B g05        50 members | band_ok ALL | dup 1 | time-limited 0 |   0.5 min | max ham 10,084
B guard_g05  50 members | band_ok ALL | dup 1 | time-limited 0 |   0.8 min | max ham 10,084
B g02        50 members | band_ok ALL | dup 0 | time-limited 0 |   1.1 min | max ham 9,408
